⏱️ **Time required:** ~5 minutes | **Type:** Hands-on quickstart

# LakeLogic — 5 Minute Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/00_quickstart.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/00_quickstart.ipynb)

One contract. One pipeline. Every row accounted for. Five minutes.

In [4]:
pip install lakelogic

Note: you may need to restart the kernel to use updated packages.


c:\_Personal\_SaaS\lakelogic\.venv\Scripts\python.exe: No module named pip


In [3]:
# Install lakelogic
#!pip install -q lakelogic[polars,duckdb]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

lakelogic v1.26.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


### ⚙️ Execution Engine

In [23]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "polars"  # 'polars' , 'duckdb', 'spark'

## The Problem

You have raw order data landing in your lake. Some rows have bad emails, negative amounts, or unknown statuses. You need to validate every row, quarantine the bad ones, and prove nothing was silently dropped — with zero custom Python logic.

## The Solution

In [24]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: orders

info:
  title: E-Commerce Orders
  version: 1.0.0
  owner: data-team@company.com
  target_layer: silver

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_email
      type: string
      required: true
      pii: true
    - name: amount
      type: float
      required: true
    - name: currency
      type: string
    - name: status
      type: string
    - name: created_at
      type: string

transformations:
  - phase: "post"
    derive:
      field: "amount_gbp"
      sql: "CAST(CASE WHEN currency='USD' THEN amount*0.79 WHEN currency='EUR' THEN amount*0.86 ELSE amount END AS DECIMAL(10,2))"

quality:
  row_rules:
    - name: valid_email
      sql: "customer_email LIKE '%@%.%'"
    - name: positive_amount
      sql: "amount > 0"
    - name: valid_status
      sql: "status IN ('pending','shipped','delivered','returned')"
    - name: valid_currency
      sql: "currency IN ('GBP','USD','EUR')"
    - name: valid_order_id
      sql: "order_id > 0"

""",
    "00_quickstart_demo/orders_contract.yaml",
)

# Generate 1000 rows — 10% intentionally bad
source_df = ll.DataGenerator(contract).generate(rows=1000, invalid_ratio=0.10, output_format=ENGINE)

# Run the pipeline
proc = ll.DataProcessor(contract, engine=ENGINE)
good, bad = proc.run(source_df)
good, bad = s.to_polars(good), s.to_polars(bad)

2026-05-10 12:59:10.249 | INFO     | lakelogic.core.generator:generate:3422 - 📋 Generating data for: E-Commerce Orders
2026-05-10 12:59:10.250 | INFO     | lakelogic.core.generator:generate:3423 -    Records    : 900 valid + 100 invalid = 1,000 total
2026-05-10 12:59:10.250 | INFO     | lakelogic.core.generator:generate:3439 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-05-10 12:59:10.251 | INFO     | lakelogic.core.generator:generate:3454 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-05-10 12:59:10.326 | INFO     | lakelogic.core.generator:generate:3488 -    Row generation complete: 1,000 records built
2026-05-10 12:59:10.326 | INFO     | lakelogic.core.generator:generate:3510 -    Test cases : 217 across 7 categories
2026-05-10 12:59:10.328 | INFO     | lakelogic.core.generator:generate:3512 -      NOT_NULL_VIOLATION               82 injections
2026-05-10 12:59:10.328 | INFO     | lakelogic.core.generator:generate:3512 -      EMPTY_STR

## The Proof

In [25]:
# Every row accounted for
s.assert_reconciliation(source_df, good, bad)

source=1000  good=877  bad=123
1000 == 877 + 123 -> True


In [23]:
# What was caught
print("Quarantined rows (sample):")
display(bad.head(5))

Quarantined rows (sample):


order_id,customer_email,amount,currency,status,created_at,_is_invalid,_test_case_types,amount_gbp,_lakelogic_errors,quarantine_state,quarantine_reprocessed
i64,str,f64,str,str,str,bool,str,"decimal[10,2]",list[str],str,bool
-849,"""euufbgp@ead.io""",19.63,"""USD""","""pending""","""2026-03-13T19:51:29.837884""",true,"""RANGE_VIOLATION""",15.51,"[""Rule failed: valid_order_id (order_id > 0)""]","""active""",false
8050,"""jhqhlx@tosozq.org""",null,"""GBP""","""returned""","""2026-04-06T19:21:31.842095""",true,"""NOT_NULL_VIOLATION""",null,"[""Rule failed: amount_required (""amount"" IS NOT NULL)"", ""Rule failed: positive_amount (amount > 0)""]","""active""",false
4698,"""oepni@xmtrx.net""",52.54,"""""",null,null,true,"""EMPTY_STRING,NOT_NULL_VIOLATION""",52.54,"[""Rule failed: valid_status (status IN ('pending','shipped','delivered','returned'))"", ""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","""active""",false
2847,"""zjybev@kzh.io""",28.59,null,"""delivered""","""2026-02-13T16:56:10.776106""",false,null,28.59,"[""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","""active""",false
4849,"""xjlnsvzygy@uebn.org""",16.31,null,"""returned""","""2026-03-02T19:53:49.803082""",false,null,16.31,"[""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","""active""",false


In [24]:
# What was good
print("Valid rows (sample):")
display(good.head(5))

Valid rows (sample):


order_id,customer_email,amount,currency,status,created_at,_is_invalid,_test_case_types,amount_gbp
i64,str,f64,str,str,str,bool,str,"decimal[10,2]"
9482,"""gmgmhn@cnr.net""",11.11,"""EUR""","""returned""","""2026-02-15T13:32:23.826237""",false,null,9.55
2312,"""jzboupzag@nytkxvrm.com""",133.11,"""GBP""","""returned""","""2026-04-08T08:57:57.813659""",false,null,133.11
9129,"""cdzsij@bhr.co""",36.67,"""EUR""","""shipped""","""2026-02-13T09:27:39.819669""",false,null,31.54
6975,"""aljvythqrm@alvscj.io""",69.74,"""USD""","""shipped""","""2026-05-03T02:09:29.803082""",false,null,55.09
1357,"""wnvqmpraa@lqbd.net""",37.13,"""USD""","""returned""","""2026-04-07T11:32:02.820671""",false,null,29.33


In [25]:
# Full audit trail
s.print_report(proc)

Run ID      : ec3e41ea-1fae-498e-8963-d1a7543cb63a
Timestamp   : 2026-05-06T13:11:09+00:00
Source      : 1000
Good        : 881
Quarantined : 119

Rule failures:
  valid_order_id: 38 rows
  amount_required: 20 rows
  positive_amount: 32 rows
  valid_status: 32 rows
  valid_currency: 69 rows
  valid_email: 37 rows
  order_id_required: 15 rows
  customer_email_required: 10 rows


{'run_id': 'ec3e41ea-1fae-498e-8963-d1a7543cb63a',
 'pipeline_run_id': None,
 'engine': 'duckdb',
 'contract': 'E-Commerce Orders',
 'contract_file_name': None,
 'contract_version': '1.0.0',
 'stage': 'default',
 'dataset': 'orders',
 'domain': None,
 'system': None,
 'environment': 'local',
 'data_layer': 'silver',
 'source_path': None,
 'source_files': [],
 'max_source_mtime': None,
 'timestamp': '2026-05-06T13:11:09+00:00',
 'counts': {'source': 1000,
  'total': 1000,
  'good': 881,
  'quarantined': 119,
  'quarantine_ratio': 0.119,
  'pre_transform_dropped': 0},
 'dataset_rules': [],
 'slos': {},
 'row_rule_failures': [{'name': 'valid_order_id',
   'sql': 'order_id > 0',
   'message': 'Rule failed: valid_order_id (order_id > 0)',
   'count': 38},
  {'name': 'amount_required',
   'sql': '"amount" IS NOT NULL',
   'message': 'Rule failed: amount_required ("amount" IS NOT NULL)',
   'count': 20},
  {'name': 'positive_amount',
   'sql': 'amount > 0',
   'message': 'Rule failed: positiv

## What You Just Saw

- **One YAML contract** defined schema, quality rules, and a derived column
- **100% reconciliation** — source == good + bad, every row accounted for
- **Automatic audit trail** — run ID, timestamp, per-rule failure counts

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.